# 📊 Análise de Padrões Históricos 2012-2026
## PriceAction + Confluências (WIN, WINFUT, WDO, WDOFUT, WINJ26, WDOJ26)

**Objetivo**: Identificar padrões recorrentes, tendências, sazonalidade e anomalias.  
**Períodos**: 2012-2014 | 2014-2016 | 2016-2018 | 2018-2020 | 2020-2022 | 2022-2024 | 2024-2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configurar visualizações
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Bibliotecas importadas com sucesso!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

## 1️⃣ Carregar Dados Históricos (2012-2026)

In [ ]:
base_path = Path("e:/repo/PriceAction_Fisica/WESQUAD/DadosCandlesBacktest")
folders = ['2012_14', '2014_16', '2016_18', '2018_20', '2020_22', '2022_24', '2024_26']

all_data = {}
metadata = {}

for folder in folders:
    folder_path = base_path / folder
    csv_files = list(folder_path.glob("*.csv"))
    
    print(f"\n📁 Período {folder}:")
    print(f"   {len(csv_files)} arquivos encontrados")
    
    dfs = []
    file_list = []
    
    for csv_file in sorted(csv_files):
        try:
            df = pd.read_csv(csv_file, sep=';', decimal=',')
            df['ArquivoOrigem'] = csv_file.stem
            dfs.append(df)
            file_list.append(csv_file.name)
        except Exception as e:
            print(f"   ⚠️  Erro em {csv_file.name}: {e}")
    
    if dfs:
        combined = pd.concat(dfs, ignore_index=True)
        all_data[folder] = combined
        
        metadata[folder] = {
            'Total Linhas': len(combined),
            'Ativos Únicos': combined['Ativo'].nunique(),
            'Ativos': combined['Ativo'].unique().tolist(),
            'Arquivos': len(file_list)
        }
        
        print(f"   ✅ {len(combined):,} candles carregados")
        print(f"   ✅ Ativos: {', '.join(combined['Ativo'].unique())}")

print("\n" + "="*80)
print("RESUMO DE DADOS CARREGADOS")
print("="*80)

metadata_df = pd.DataFrame(metadata).T
print(metadata_df[['Total Linhas', 'Ativos Únicos', 'Arquivos']])

## 2️⃣ Limpeza e Preparação dos Dados

In [ ]:
def limpar_dados(df):
    """Limpa e converte tipos de dados"""
    df_clean = df.copy()
    
    # Remover espaços nas colunas
    df_clean.columns = df_clean.columns.str.strip()
    
    # Converter data e hora
    df_clean['Data'] = pd.to_datetime(df_clean['Data'], format='%d/%m/%Y', errors='coerce')
    df_clean['Hora'] = pd.to_datetime(df_clean['Hora'], format='%H:%M:%S', errors='coerce').dt.time
    
    # Colunas numéricas
    numeric_cols = ['Abertura', 'Máximo', 'Mínimo', 'Fechamento', 'Volume', 'Quantidade']
    for col in numeric_cols:
        if col in df_clean.columns:
            # Remover pontos (separador de milhares) e substituir vírgula por ponto
            df_clean[col] = df_clean[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    # Remover linhas com valores nulos críticos
    df_clean = df_clean.dropna(subset=['Data', 'Fechamento'])
    
    return df_clean

# Limpar todos os dados
all_data_clean = {}
for folder, df in all_data.items():
    all_data_clean[folder] = limpar_dados(df)
    print(f"✅ {folder}: {len(all_data_clean[folder]):,} candles após limpeza")

# Criar dataframe consolidado
df_consolidated = pd.concat(all_data_clean.values(), ignore_index=True)
df_consolidated = df_consolidated.sort_values('Data').reset_index(drop=True)

print(f"\n✅ DataFrame consolidado: {len(df_consolidated):,} candles")
print(f"\nPrimeiras linhas:")
print(df_consolidated.head())

## 3️⃣ Análise Exploratória: Estatísticas por Período

In [ ]:
print("="*100)
print("ESTATÍSTICAS POR PERÍODO (2012-2026)")
print("="*100)

stats_periodo = []

for folder, df in all_data_clean.items():
    if len(df) > 0:
        preco_medio = df['Fechamento'].mean()
        volatilidade = df['Fechamento'].pct_change().std() * 100
        volume_medio = df['Volume'].mean()
        max_price = df['Máximo'].max()
        min_price = df['Mínimo'].min()
        range_price = max_price - min_price
        
        stats_periodo.append({
            'Período': folder,
            'Candles': len(df),
            'Preço Médio': f"{preco_medio:.2f}",
            'Volatilidade (%)': f"{volatilidade:.2f}",
            'Volume Médio': f"{volume_medio:,.0f}",
            'Max': f"{max_price:.2f}",
            'Min': f"{min_price:.2f}",
            'Range': f"{range_price:.2f}"
        })

stats_df = pd.DataFrame(stats_periodo)
print(stats_df.to_string(index=False))

# Gráfico: Evolução de Volatilidade
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

volatilidades = []
periodos = []
for folder, df in all_data_clean.items():
    if len(df) > 0:
        vol = df['Fechamento'].pct_change().std() * 100
        volatilidades.append(vol)
        periodos.append(folder)

axes[0, 0].bar(periodos, volatilidades, color='steelblue')
axes[0, 0].set_title('Volatilidade por Período', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Volatilidade (%)')
axes[0, 0].grid(alpha=0.3)

# Preço médio por período
precos_medios = []
for folder, df in all_data_clean.items():
    if len(df) > 0:
        precos_medios.append(df['Fechamento'].mean())

axes[0, 1].plot(periodos, precos_medios, marker='o', linewidth=2, markersize=8, color='green')
axes[0, 1].set_title('Preço Médio por Período', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Preço Médio')
axes[0, 1].grid(alpha=0.3)

# Volume médio por período
volumes_medios = []
for folder, df in all_data_clean.items():
    if len(df) > 0:
        volumes_medios.append(df['Volume'].mean())

axes[1, 0].bar(periodos, volumes_medios, color='coral')
axes[1, 0].set_title('Volume Médio por Período', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Volume Médio')
axes[1, 0].grid(alpha=0.3)

# Distribuição de ativos
ativos_count = {}
for folder, df in all_data_clean.items():
    for ativo in df['Ativo'].unique():
        if ativo not in ativos_count:
            ativos_count[ativo] = 0
        ativos_count[ativo] += len(df[df['Ativo'] == ativo])

axes[1, 1].barh(list(ativos_count.keys()), list(ativos_count.values()), color='mediumpurple')
axes[1, 1].set_title('Total de Candles por Ativo', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Total de Candles')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Análise Exploratória Completa!")

## 4️⃣ Padrões de Price Action: Análise por Ativo

In [ ]:
def calcular_metricas_pa(df):
    """Calcula métricas de Price Action"""
    metricas = {}
    
    for ativo in df['Ativo'].unique():
        df_ativo = df[df['Ativo'] == ativo].sort_values('Data').reset_index(drop=True)
        
        if len(df_ativo) < 2:
            continue
        
        # Taxa de ganho (candles com fechamento > abertura)
        df_ativo['GainBar'] = (df_ativo['Fechamento'] > df_ativo['Abertura']).astype(int)
        taxa_ganho = df_ativo['GainBar'].mean() * 100
        
        # Amplitude média (high - low)
        df_ativo['Amplitude'] = df_ativo['Máximo'] - df_ativo['Mínimo']
        amplitude_media = df_ativo['Amplitude'].mean()
        
        # Corpo do candle (close - open)
        df_ativo['Corpo'] = abs(df_ativo['Fechamento'] - df_ativo['Abertura'])
        corpo_medio = df_ativo['Corpo'].mean()
        
        # Ratio finsternis (amplitude / corpo)
        df_ativo['RatioFinsternis'] = np.where(df_ativo['Corpo'] != 0, df_ativo['Amplitude'] / df_ativo['Corpo'], np.nan)
        ratio_medio = df_ativo['RatioFinsternis'].mean()
        
        metricas[ativo] = {
            'Taxa Ganho (%)': taxa_ganho,
            'Amplitude Médio': amplitude_media,
            'Corpo Médio': corpo_medio,
            'Ratio Amplitude/Corpo': ratio_medio,
            'Total Candles': len(df_ativo)
        }
    
    return metricas

metricas_pa = calcular_metricas_pa(df_consolidated)

print("\n" + "="*100)
print("PADRÕES DE PRICE ACTION POR ATIVO")
print("="*100)

pa_df = pd.DataFrame(metricas_pa).T
print(pa_df.round(4))

# Visualizar padrões
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ativos = list(metricas_pa.keys())
taxa_ganhos = [metricas_pa[a]['Taxa Ganho (%)'] for a in ativos]
amplitudes = [metricas_pa[a]['Amplitude Médio'] for a in ativos]
corpos = [metricas_pa[a]['Corpo Médio'] for a in ativos]
ratios = [metricas_pa[a]['Ratio Amplitude/Corpo'] for a in ativos]

axes[0, 0].bar(ativos, taxa_ganhos, color='green', alpha=0.7)
axes[0, 0].set_title('Taxa de Ganho (%) por Ativo', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('% de Candles com Fechamento > Abertura')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].bar(ativos, amplitudes, color='steelblue', alpha=0.7)
axes[0, 1].set_title('Amplitude Média por Ativo', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Amplitude (High - Low)')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].bar(ativos, corpos, color='coral', alpha=0.7)
axes[1, 0].set_title('Corpo Médio do Candle por Ativo', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Corpo (|Close - Open|)')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].bar(ativos, ratios, color='mediumpurple', alpha=0.7)
axes[1, 1].set_title('Ratio Finsternis (Amplitude/Corpo)', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Ratio')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Análise de Price Action Concluída!")

## 5️⃣ Padrões Detectados: Insights Principais

In [ ]:
print("\n" + "="*100)
print("🔍 INSIGHTS E PADRÕES DETECTADOS")
print("="*100)

# 1. Ativo mais volátil
volatilidades_ativo = {}
for ativo in df_consolidated['Ativo'].unique():
    df_ativo = df_consolidated[df_consolidated['Ativo'] == ativo]
    vol = df_ativo['Fechamento'].pct_change().std() * 100
    volatilidades_ativo[ativo] = vol

ativo_mais_volatil = max(volatilidades_ativo, key=volatilidades_ativo.get)
ativo_menos_volatil = min(volatilidades_ativo, key=volatilidades_ativo.get)

print(f"\n📈 VOLATILIDADE:")
print(f"   • Ativo MAIS VOLÁTIL: {ativo_mais_volatil} ({volatilidades_ativo[ativo_mais_volatil]:.2f}%)")
print(f"   • Ativo MENOS VOLÁTIL: {ativo_menos_volatil} ({volatilidades_ativo[ativo_menos_volatil]:.2f}%)")

# 2. Períodos de maior movimento
print(f"\n📊 MOVIMENTO POR PERÍODO:")
for folder in sorted(all_data_clean.keys()):
    df = all_data_clean[folder]
    if len(df) > 0:
        retorno_total = ((df['Fechamento'].iloc[-1] - df['Fechamento'].iloc[0]) / df['Fechamento'].iloc[0]) * 100
        amplitude_media = (df['Máximo'] - df['Mínimo']).mean()
        print(f"   • {folder}: Retorno {retorno_total:+.2f}%, Amplitude Média {amplitude_media:.2f}")

# 3. Padrões de Price Action mais comuns
print(f"\n🎯 PADRÕES DE PRICE ACTION:")
for ativo in metricas_pa.keys():
    taxa_ganho = metricas_pa[ativo]['Taxa Ganho (%)']
    if taxa_ganho > 52:
        bias = "BULLISH"
    elif taxa_ganho < 48:
        bias = "BEARISH"
    else:
        bias = "NEUTRO"
    
    print(f"   • {ativo}: {taxa_ganho:.1f}% ganhos ({bias})")

# 4. Recomendações
print(f"\n💡 RECOMENDAÇÕES PARA ESTRATÉGIA:")
print(f"   1. Focar em {ativo_mais_volatil}: maior amplitude = mais oportunidades")
print(f"   2. Usar stops curtos em {ativo_menos_volatil}: baixa volatilidade")
print(f"   3. Analisar confluências (Volume + Price Action + Tendência)")
print(f"   4. Testar estratégias em cada período isoladamente")
print(f"   5. Validar em backtest com dados históricos completos")

print("\n" + "="*100)